# Pré Annotation dates et ajout sur fichier BRAT

### Import

In [ ]:
%load_ext autoreload
%load_ext nb_black
%autoreload 2

In [ ]:
from copy import deepcopy
from itertools import chain, repeat
from typing import Callable, Iterable, List, Dict, Optional
import sys
import os
import pandas as pd
import torch
from confit import Cli
from pydantic import DirectoryPath
from spacy import displacy
from spacy.tokens import Doc, Span
from tqdm import tqdm
from spacy import Language
from confit.utils.random import set_seed
from transformers import AutoTokenizer, AutoModel
import edsnlp, edsnlp.pipes as eds
from edsnlp import registry, Pipeline
from edsnlp.scorers.ner import create_ner_exact_scorer
from torch import Tensor
import matplotlib.pyplot as plt
from datetime import datetime
import datetime
import time
from edsnlp.connectors.brat import BratConnector

### Dossier source BRAT

In [ ]:
dossier = "/home/pidoux/LIMICS/brat/data/RENE/"

### Import des docs et entités TRUE

In [ ]:
doc_iterator = edsnlp.data.read_standoff(
    dossier,
    span_setter={"ents": "Temporal"},
)
true_docs = list(doc_iterator)

In [ ]:
#displacy.render(true_docs[0], style="ent")

### Import des docs et prédictions des entités dates avec EDS-NLP pipeline PRED

In [ ]:
def txt_liste(dossier:str):
    corpus = []
    name=[]
    for fichier in os.listdir(dossier):
        chemin = os.path.join(dossier, fichier)
        if os.path.isfile(chemin) and fichier.endswith('.txt'):
            with open(chemin, 'r', encoding='utf-8') as f:
                corpus.append(f.read())
                name.append(fichier[:-4])
    return corpus,name

In [ ]:
corpus,name = txt_liste(dossier)
docs = edsnlp.data.from_iterable(corpus)

In [ ]:
nlp = edsnlp.blank("eds")
nlp.add_pipe(eds.sentences())
nlp.add_pipe(eds.normalizer())
nlp.add_pipe(eds.dates()) 
pred_iterator = docs.map_pipeline(nlp)
pred_docs = list(pred_iterator)
#displacy.render(pred_docs[2], style="span", options={"spans_key": "dates"} )

In [ ]:
#pred_docs[0].ents = pred_docs[0].spans["dates"]
#displacy.render(pred_docs, style="ent")

In [ ]:
print(pred_docs[0].spans)

In [ ]:
dossier2 = "../../brat/data/test/"

brat = BratConnector(dossier2)
for i,doc in enumerate(pred_docs):
    doc._.note_id = name[i]
    doc.spans["pollutions"] = []
br = brat.docs2brat(pred_docs)

In [ ]:
dossier3 = "../../brat/data/merge/"

# Lire le premier fichier
for nom_fichier in name:
    with open(dossier+nom_fichier+'.ann', 'r', encoding='utf-8') as file:
        lines_file1 = file.readlines()

    # Extraire les identifiants existants
    existing_ids = max([int(line.split('\t')[0][1:]) for line in lines_file1])

    # Lire le second fichier et préparer les nouvelles lignes
    new_lines = []
    with open(dossier2+nom_fichier+'.ann', 'r', encoding='utf-8') as file:
        for line in file:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue  # Skip les lignes mal formées
            # Remplacer 'date' par 'Temporal'
            if 'date' in parts[1]:
                parts[1] = parts[1].replace('date', 'Temporal')
            # Générer un nouvel identifiant si 
            parts[0] = 'T'+str(existing_ids+1)
            existing_ids += 1
            new_line = '\t'.join(parts)+'\n'
            new_lines.append(new_line)

    # Fusionner les lignes du premier fichier avec les nouvelles lignes
    merged_lines = lines_file1 + new_lines

    # Écrire le fichier fusionné
    with open(dossier3+nom_fichier+'.ann', 'w', encoding='utf-8') as file:
        file.writelines(merged_lines)

    print("Fusion terminée. Les annotations ont été sauvegardées dans:")
